In [ ]:
pip install keras-tuner --upgrade

In [ ]:
import pandas as pd
import numpy as np

# Ler arquivos
hmd = pd.read_csv('/content/hmd_ambos.csv')
bra = pd.read_csv('/content/tabua_ambos.csv')

hmd.rename(columns={'Year':'ANO'}, inplace=True)
ages = ['0','1','5','10','15','20','25','30','35','40','45','50','55','60','65','70','75','80','85','90']
bra = bra[['ANO']+ages]
hmd = hmd[['ANO','Country']+ages]

def extract_country(df, country):
    out = df[df.Country==country].drop(columns='Country').set_index('ANO').sort_index()
    return out[ages]

bra_mx_log = bra.set_index('ANO').sort_index()[ages]  # Brasil em log(mx)
hmd

In [ ]:
bra_mx_log

In [ ]:
common_years = np.intersect1d(hmd.ANO.unique(), bra.ANO.unique())

dist = []
for c in hmd.Country.unique():
    tmp = extract_country(hmd, c)
    if not set(common_years).issubset(tmp.index):
        continue
    d = np.abs(tmp.loc[common_years, ages].values -
               bra_mx_log.loc[common_years, ages].values).mean()
    dist.append((c, d))

sim_df = pd.DataFrame(dist, columns=['country','manhattan']).sort_values('manhattan')

K = 10
alpha = 1
topK = sim_df.head(K)
topK['weight'] = np.exp(-alpha*topK.manhattan)
topK['weight'] /= topK.weight.sum()

# Exibir países e pesos
print("Países selecionados (mais aderentes primeiro):")
print(topK[['country', 'weight']].sort_values('weight', ascending=False).reset_index(drop=True))

# Salvar em CSV (opcional)
topK[['country', 'weight']].to_csv('paises_selecionados_pesos.csv', index=False)

# Gráfico de barras dos pesos
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.bar(topK.sort_values('weight', ascending=False)['country'],
        topK.sort_values('weight', ascending=False)['weight'],
        color='royalblue')
plt.ylabel('Peso')
plt.xlabel('País')
plt.title('Pesos dos países mais similares ao Brasil')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
seq_len = 5
X_train, y_train, sw = [], [], []
for _,row in topK.iterrows():
    c = row.country; w = row.weight
    mx = extract_country(hmd, c).loc[:2015]
    # Normalização
    mx_norm = (mx - mx.mean()) / mx.std()
    for i in range(len(mx_norm)-seq_len):
        X_train.append(mx_norm.iloc[i:i+seq_len].values[...,None])
        y_train.append(mx_norm.iloc[i+seq_len].values)
        sw.append(w)
X_train = np.array(X_train)
y_train = np.array(y_train)
sw = np.array(sw)

In [ ]:
import keras_tuner as kt
from tensorflow import keras
import numpy as np

# --- Construção do modelo CNN original (com correção de kernel_size/padding) ---
def build_model(hp):
    # nunca deixa o kernel_size maior que seq_len
    max_kernel = min(5, seq_len)

    model = keras.Sequential([
        keras.layers.Conv1D(
            filters=hp.Int('filters', 16, 64, step=16),
            kernel_size=hp.Int('kernel_size', 2, max_kernel),
            activation='relu',
            padding='same',    # evita redução negativa
            input_shape=(seq_len, len(ages))
        ),
        keras.layers.Dropout(hp.Float('dropout_cnn', 0.1, 0.3, step=0.1)),

        keras.layers.Flatten(),
        keras.layers.Dense(
            units=hp.Int('dense_units', 32, 128, step=32),
            activation='relu'
        ),
        keras.layers.Dropout(hp.Float('dropout_dense', 0.1, 0.3, step=0.1)),

        keras.layers.Dense(len(ages), activation='linear')
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Float('learning_rate', 1e-4, 1e-2, sampling='log')
        ),
        loss='mse',
        metrics=['mae']
    )
    return model

# --- Tuner ---
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=10,
    directory='tune_cnn_only'
)

tuner.search(
    X_train, y_train,
    validation_split=0.1,
    sample_weight=sw,
    epochs=500,
    batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

base_model = tuner.get_best_models(1)[0]
base_model.save('cnn_only_finetuned_brasil.keras')

# --- Preparação dos dados brasileiros ---
mean_bra = bra_mx_log.loc[:2015].mean()
std_bra  = bra_mx_log.loc[:2015].std()
mx_bra_norm = (bra_mx_log - mean_bra) / std_bra

Xb, yb = [], []
for i in range(len(mx_bra_norm) - seq_len - 4):
    Xb.append(mx_bra_norm.iloc[i:i+seq_len].values[..., None])
    yb.append(mx_bra_norm.iloc[i+seq_len].values)
Xb = np.array(Xb)
yb = np.array(yb)

# --- Fine-tuning em duas fases ---
print("Iniciando fine-tuning Brasil...")

# Fase 1: congela convoluções, treina apenas densas
for layer in base_model.layers:
    if 'conv' in layer.name or 'batch' in layer.name:
        layer.trainable = False
    else:
        layer.trainable = True

base_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
base_model.fit(
    Xb, yb,
    epochs=500,
    batch_size=32,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)]
)

# Fase 2: libera todas as camadas com LR menor
for layer in base_model.layers:
    layer.trainable = True

base_model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='mse', metrics=['mae'])
base_model.fit(
    Xb, yb,
    epochs=500,
    batch_size=32,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)]
)

# Model final
base_model.save('cnn_finetuned_brasil.keras')
print("Modelo final salvo como cnn_finetuned_brasil.keras")

In [ ]:
# Carregar modelo
from tensorflow import keras
base_model = keras.models.load_model('cnn_finetuned_brasil (1).keras')
base_model.summary()

In [ ]:
# 6.1 Previsão recursiva (normalizada)

# --- Preparação dos dados brasileiros ---
mean_bra = bra_mx_log.loc[:2015].mean()
std_bra  = bra_mx_log.loc[:2015].std()
mx_bra_norm = (bra_mx_log - mean_bra) / std_bra

Xb, yb = [], []
for i in range(len(mx_bra_norm) - seq_len - 4):
    Xb.append(mx_bra_norm.iloc[i:i+seq_len].values[..., None])
    yb.append(mx_bra_norm.iloc[i+seq_len].values) # Corrected from mx_norm to mx_bra_norm
Xb = np.array(Xb)
yb = np.array(yb)

mx_pred_norm = mx_bra_norm.copy()
for year in range(2016, 2020):
    window = mx_pred_norm.loc[year-seq_len:year-1].values[...,None]
    pred = base_model.predict(window[np.newaxis,:,:,:])[0]
    mx_pred_norm.loc[year] = pred

# 6.2 Desnormalizar para log(mx) e converter para mx
mx_pred_log = mx_pred_norm.loc[2016:2019] * std_bra.values + mean_bra.values
mx_pred = np.exp(mx_pred_log)

# 6.3 Bootstrap CORRIGIDO para IC 95% (em mx)
B = 1000
boot = np.empty((B, 4, len(ages)))
resid = (yb - base_model.predict(Xb))  # resíduos na escala normalizada

# Garantir que os resíduos tenham média zero (opcional, mas recomendado)
resid = resid - resid.mean(axis=0)

for b in range(B):
    # Para cada réplica bootstrap
    boot_pred_norm = mx_pred_norm.loc[2016:2019].values.copy()

    # Adicionar resíduos reamostrados para cada ano individualmente
    for i in range(4):  # 4 anos (2016-2019)
        # Reamostra um resíduo aleatório para cada ano
        idx_resid = np.random.choice(resid.shape[0], size=1, replace=True)
        boot_pred_norm[i, :] += resid[idx_resid[0], :]

    # Desnormalizar para log(mx) e converter para mx
    boot_pred_log = boot_pred_norm * std_bra.values + mean_bra.values
    boot[b] = np.exp(boot_pred_log)

# Calcular ICs
lower = np.percentile(boot, 2.5, axis=0)
upper = np.percentile(boot, 97.5, axis=0)

# 6.4 Valores observados (se disponíveis para 2016-2019)
anos_pred = [2016, 2017, 2018, 2019]
try:
    # Se os dados observados estão disponíveis no arquivo original
    mx_obs_log = bra_mx_log.loc[2016:2019]
    mx_obs = np.exp(mx_obs_log.values)
    tem_observado = True
except KeyError:
    # Se não há dados observados para esse período
    mx_obs = np.full((4, len(ages)), np.nan)
    tem_observado = False

# 6.5 Criar DataFrame consolidado
records = []
for i, ano in enumerate(anos_pred):
    for j, idade_str in enumerate(ages): # Renomeado para evitar conflito com a variável 'idade' no dict
        idade = int(idade_str) if idade_str.isdigit() else idade_str # Converte a idade para int se for um dígito

        # Acesse os valores usando os índices numéricos i e j, pois mx_obs, mx_pred, lower e upper são arrays numpy
        observado_val = mx_obs[i, j] if tem_observado else np.nan
        previsto_val = mx_pred.iloc[i, j] # Use iloc para indexação baseada em posição no DataFrame
        ic_lower_val = lower[i, j]
        ic_upper_val = upper[i, j]

        records.append({
            'ano': ano,
            'idade': idade,
            'observado': observado_val,
            'previsto': previsto_val,
            'IC_lower': ic_lower_val,
            'IC_upper': ic_upper_val
        })

df_previsoes = pd.DataFrame(records)

# 6.6 Salvar resultados
df_previsoes.to_csv('previsoes_cnn_gru_2016_2019.csv', index=False)
print(f"DataFrame criado com {len(df_previsoes)} registros")
print(f"Colunas: {list(df_previsoes.columns)}")
print(f"Anos: {df_previsoes.ano.unique()}")
print(f"Idades: {df_previsoes.idade.unique()}")

# 6.7 Visualizar amostra
print("\nPrimeiras 10 linhas:")
print(df_previsoes.head(10))

In [ ]:
# Previsão no teste
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# Filtra apenas anos de teste e linhas com observado disponível
anos_teste = [2016, 2017, 2018, 2019]
df_test = df_previsoes[df_previsoes['ano'].isin(anos_teste) & df_previsoes['observado'].notna()]

# Cálculo das métricas por idade
def calc_metrics(g):
    rmse = np.sqrt(mean_squared_error(g['observado'], g['previsto']))
    mae = mean_absolute_error(g['observado'], g['previsto'])
    smape = (200 * np.abs(g['observado'] - g['previsto']) / (np.abs(g['observado']) + np.abs(g['previsto']))).mean()
    return pd.Series({'RMSE': rmse, 'MAE': mae, 'sMAPE': smape})

metrics = df_test.groupby('idade').apply(calc_metrics).reset_index()

# Cálculo das métricas globais (todas as idades e anos juntos)
rmse_global = np.sqrt(mean_squared_error(df_test['observado'], df_test['previsto']))
mae_global = mean_absolute_error(df_test['observado'], df_test['previsto'])
smape_global = (200 * np.abs(df_test['observado'] - df_test['previsto']) / (np.abs(df_test['observado']) + np.abs(df_test['previsto']))).mean()

print('Métricas globais no teste (2016-2019):')
print(f'RMSE: {rmse_global:.4f}')
print(f'MAE: {mae_global:.4f}')
print(f'sMAPE: {smape_global:.2f}%')

# Salva as métricas por idade
metrics.to_csv('metricas_teste_2016_2019.csv', index=False)
metrics

In [ ]:
metrics[['RMSE', 'MAE', 'sMAPE']].mean()

In [ ]:
# --- Normalização até 2011 ---
mean_bra = bra_mx_log.loc[:2011].mean()  # Média até 2011
std_bra = bra_mx_log.loc[:2011].std()    # Desvio padrão até 2011
mx_bra_norm = (bra_mx_log - mean_bra) / std_bra

# --- Previsão Recursiva 2012-2015 ---
mx_pred_norm = mx_bra_norm.copy()
for year in range(2012, 2016):  # 2012 a 2015
    window = mx_pred_norm.loc[year-seq_len:year-1].values[..., None]
    pred = base_model.predict(window[np.newaxis, :, :, :])[0]
    mx_pred_norm.loc[year] = pred

# Desnormalizar
mx_pred_log = mx_pred_norm.loc[2012:2015] * std_bra.values + mean_bra.values
mx_pred = np.exp(mx_pred_log)

# --- Bootstrap com Resíduos até 2011 ---
Xb, yb = [], []
for i in range(len(mx_bra_norm.loc[:2011]) - seq_len - 4):  # Dados até 2011
    Xb.append(mx_bra_norm.iloc[i:i+seq_len].values[..., None])
    yb.append(mx_bra_norm.iloc[i+seq_len].values)
Xb = np.array(Xb)
yb = np.array(yb)

resid = (yb - base_model.predict(Xb))  # Resíduos até 2011
resid = resid - resid.mean(axis=0)     # Centralizar resíduos

B = 1000
boot = np.empty((B, 4, len(ages)))
for b in range(B):
    boot_pred_norm = mx_pred_norm.loc[2012:2015].values.copy()
    for i in range(4):  # 4 anos (2012-2015)
        idx_resid = np.random.choice(resid.shape[0], size=1, replace=True)
        boot_pred_norm[i, :] += resid[idx_resid[0], :]
    boot_pred_log = boot_pred_norm * std_bra.values + mean_bra.values
    boot[b] = np.exp(boot_pred_log)

# Intervalos de confiança
lower = np.percentile(boot, 2.5, axis=0)
upper = np.percentile(boot, 97.5, axis=0)

# --- Métricas (2012-2015) ---
anos_pred = [2012, 2013, 2014, 2015]
mx_obs = np.exp(bra_mx_log.loc[2012:2015].values)  # Dados observados

records = []
for i, ano in enumerate(anos_pred):
    for j, idade_str in enumerate(ages):
        idade = int(idade_str) if idade_str.isdigit() else idade_str
        records.append({
            'ano': ano,
            'idade': idade,
            'observado': mx_obs[i, j],
            'previsto': mx_pred.iloc[i, j],
            'IC_lower': lower[i, j],
            'IC_upper': upper[i, j]
        })

df_previsoes = pd.DataFrame(records)

# Cálculo das métricas
df_test = df_previsoes[df_previsoes['observado'].notna()]
def calc_metrics(g):
    rmse = np.sqrt(mean_squared_error(g['observado'], g['previsto']))
    mae = mean_absolute_error(g['observado'], g['previsto'])
    smape = (200 * np.abs(g['observado'] - g['previsto']) / (np.abs(g['observado']) + np.abs(g['previsto']))).mean()
    return pd.Series({'RMSE': rmse, 'MAE': mae, 'sMAPE': smape})

metrics = df_test.groupby('idade').apply(calc_metrics).reset_index()

# Métricas globais
rmse_global = np.sqrt(mean_squared_error(df_test['observado'], df_test['previsto']))
mae_global = mean_absolute_error(df_test['observado'], df_test['previsto'])
smape_global = (200 * np.abs(df_test['observado'] - df_test['previsto']) / (np.abs(df_test['observado']) + np.abs(df_test['previsto']))).mean()

print('Métricas globais (2012-2015):')
print(f'RMSE: {rmse_global:.4f}')
print(f'MAE: {mae_global:.4f}')
print(f'sMAPE: {smape_global:.2f}%')

# Salvar resultados
metrics.to_csv('metricas_2012_2015.csv', index=False)
df_previsoes.to_csv('previsoes_2012_2015.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt

# Supondo que 'ages' contém a lista de idades únicas (20 no total)
n_linhas = 5
n_colunas = 4

# Cria a figura e os subplots
fig, axs = plt.subplots(n_linhas, n_colunas, figsize=(20, 25))
fig.suptitle('Curvas de Mortalidade Observada vs Prevista por Idade', y=1.02, fontsize=16)

# Ajusta o espaçamento entre os subplots
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# Itera sobre cada idade em 'ages' e plota no subplot correspondente
for i, idade_str in enumerate(ages): # Iterar sobre a lista 'ages'
    linha = i // n_colunas
    coluna = i % n_colunas

    # Garantir que a idade usada para filtrar seja do tipo correto (int se for um dígito)
    idade = int(idade_str) if idade_str.isdigit() else idade_str

    df_idade = df_previsoes[df_previsoes['idade'] == idade]

    ax = axs[linha, coluna]
    # Verifique se há dados para a idade antes de plotar
    if not df_idade.empty:
        ax.plot(df_idade['ano'], df_idade['observado'], label='Observado', color='blue')
        ax.plot(df_idade['ano'], df_idade['previsto'], label='Previsto', color='red', linestyle='--')
        ax.fill_between(df_idade['ano'], df_idade['IC_lower'], df_idade['IC_upper'], alpha=0.3, color='gray', label='IC 95%')

        ax.set_yscale('log')
        ax.set_title(f'Idade {idade}')
        ax.set_xlabel('Ano')
        ax.set_ylabel('Mortalidade')
        ax.legend()
    else:
        # Opcional: Ocultar subplot se não houver dados para a idade
        fig.delaxes(ax)


# Remove subplots vazios se houver menos idades do que subplots
for i in range(len(ages), n_linhas * n_colunas):
    linha = i // n_colunas
    coluna = i % n_colunas
    # Verifica se o eixo ainda não foi deletado
    if axs[linha, coluna] in fig.axes:
         fig.delaxes(axs[linha, coluna])


plt.tight_layout()
plt.show()

In [ ]:
# Gráfico da curva de mortalidade prevista vs observada para o ano de 2019
ano_escolhido = 2019
df_ano_escolhido = df_previsoes[df_previsoes['ano'] == ano_escolhido]
plt.figure(figsize=(10, 6))
plt.plot(df_ano_escolhido['idade'], df_ano_escolhido['observado'], label='Observado', color='red')
plt.plot(df_ano_escolhido['idade'], df_ano_escolhido['previsto'], label='Previsto', linestyle='--')
plt.fill_between(df_ano_escolhido['idade'], df_ano_escolhido['IC_lower'], df_ano_escolhido['IC_upper'], alpha=0.3, color='gray', label='IC 95%')
plt.yscale('log')
plt.xlabel('Idade')
plt.ylabel('log(mx)')
plt.title(f'Curva de Mortalidade Prevista vs Observada para o Ano {ano_escolhido}')
plt.legend()
plt.show()
#

In [ ]:
# 6.1 Previsão recursiva (normalizada)

# --- Preparação dos dados brasileiros ---
mean_bra = bra_mx_log.loc[:2015].mean()
std_bra  = bra_mx_log.loc[:2015].std()
mx_bra_norm = (bra_mx_log - mean_bra) / std_bra

Xb, yb = [], []
for i in range(len(mx_bra_norm) - seq_len - 4):
    Xb.append(mx_bra_norm.iloc[i:i+seq_len].values[..., None])
    yb.append(mx_bra_norm.iloc[i+seq_len].values) # Corrected from mx_norm to mx_bra_norm
Xb = np.array(Xb)
yb = np.array(yb)

mx_pred_norm = mx_bra_norm.copy()
for year in range(2016, 2019):
    window = mx_pred_norm.loc[year-seq_len:year-1].values # Remove [...,None] here
    pred = base_model.predict(window[np.newaxis,:,:])[0] # Adjust prediction input shape
    mx_pred_norm.loc[year] = pred

# 6.2 Desnormalizar para log(mx) e converter para mx
mx_pred_log = mx_pred_norm.loc[2016:2019] * std_bra.values + mean_bra.values
mx_pred = np.exp(mx_pred_log)

# 6.3 Bootstrap CORRIGIDO para IC 95% (em mx)
B = 1000
boot = np.empty((B, 4, len(ages)))
resid = (yb - base_model.predict(Xb))  # resíduos na escala normalizada

# Garantir que os resíduos tenham média zero (opcional, mas recomendado)
resid = resid - resid.mean(axis=0)

for b in range(B):
    # Para cada réplica bootstrap
    boot_pred_norm = mx_pred_norm.loc[2016:2019].values.copy()

    # Adicionar resíduos reamostrados para cada ano individualmente
    for i in range(4):  # 4 anos (2016-2019)
        # Reamostra um resíduo aleatório para cada ano
        idx_resid = np.random.choice(resid.shape[0], size=1, replace=True)
        boot_pred_norm[i, :] += resid[idx_resid[0], :]

    # Desnormalizar para log(mx) e converter para mx
    boot_pred_log = boot_pred_norm * std_bra.values + mean_bra.values
    boot[b] = np.exp(boot_pred_log)

# Calcular ICs
lower = np.percentile(boot, 2.5, axis=0)
upper = np.percentile(boot, 97.5, axis=0)

# 6.4 Valores observados (se disponíveis para 2016-2019)
anos_pred = [2016, 2017, 2018, 2019]
try:
    # Se os dados observados estão disponíveis no arquivo original
    mx_obs_log = bra_mx_log.loc[2016:2019]
    mx_obs = np.exp(mx_obs_log.values)
    tem_observado = True
except KeyError:
    # Se não há dados observados para esse período
    mx_obs = np.full((4, len(ages)), np.nan)
    tem_observado = False

# 6.5 Criar DataFrame consolidado
records = []
for i, ano in enumerate(anos_pred):
    for j, idade_str in enumerate(ages): # Renomeado para evitar conflito com a variável 'idade' no dict
        idade = int(idade_str) if idade_str.isdigit() else idade_str # Converte a idade para int se for um dígito

        # Acesse os valores usando os índices numéricos i e j, pois mx_obs, mx_pred, lower e upper são arrays numpy
        observado_val = mx_obs[i, j] if tem_observado else np.nan
        previsto_val = mx_pred.iloc[i, j] # Use iloc para indexação baseada em posição no DataFrame
        ic_lower_val = lower[i, j]
        ic_upper_val = upper[i, j]

        records.append({
            'ano': ano,
            'idade': idade,
            'observado': observado_val,
            'previsto': previsto_val,
            'IC_lower': ic_lower_val,
            'IC_upper': ic_upper_val
        })

df_previsoes = pd.DataFrame(records)

# 6.6 Salvar resultados
df_previsoes.to_csv('previsoes_cnn_gru_2016_2019.csv', index=False)
print(f"DataFrame criado com {len(df_previsoes)} registros")
print(f"Colunas: {list(df_previsoes.columns)}")
print(f"Anos: {df_previsoes.ano.unique()}")
print(f"Idades: {df_previsoes.idade.unique()}")

# 6.7 Visualizar amostra
print("\nPrimeiras 10 linhas:")
print(df_previsoes.head(10))